In [ ]:
# Imports
import pandas as pd
import numpy as np
import os
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, GlobalAveragePooling2D, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

In [ ]:
# Load data
train_df = pd.read_csv('/kaggle/input/bttai-ajl-2025/train.csv')
test_df = pd.read_csv('/kaggle/input/bttai-ajl-2025/test.csv')

# Add .jpg extension to md5hash column to reference the file_name
train_df['md5hash'] = train_df['md5hash'].astype(str) + '.jpg'
test_df['md5hash'] = test_df['md5hash'].astype(str) + '.jpg'

# Combine label and md5hash to form the correct path
train_df['file_path'] = train_df['label'] + '/' + train_df['md5hash']
test_df['file_path'] = test_df['md5hash']

In [ ]:
# Check the first few rows to understand the structure
print(train_df.head())
print(test_df.head())

In [ ]:
# Encode the labels
label_encoder = LabelEncoder()
train_df['encoded_label'] = label_encoder.fit_transform(train_df['label'])

# Split the data into training and validation sets
train_data, val_data = train_test_split(train_df, test_size=0.2, random_state=42)

# Define image data generators for training and validation
train_datagen = ImageDataGenerator(rescale=1./255)
val_datagen = ImageDataGenerator(rescale=1./255)

# Define the directory paths
train_dir = '/kaggle/input/bttai-ajl-2025/train/train/'

In [ ]:
def create_generator(dataframe, directory, batch_size=32, target_size=(128, 128)):
    """
    Create image generator for training or validation data.
    """
    # Fill in the correct flow_from_dataframe parameters
    generator = train_datagen.flow_from_dataframe(
        dataframe=dataframe,
        directory=directory,
        x_col='file_path', # Use combined path
        y_col='label',
        target_size=target_size,
        batch_size=batch_size,
        class_mode='categorical',
        validate_filenames=False # Disable strict filename validation
    )
    return generator

In [ ]:
# Create generators for training and validation data
train_generator = create_generator(train_data, train_dir)
val_generator = create_generator(val_data, train_dir)

In [ ]:
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(128, 128, 3)),
    BatchNormalization(),
    MaxPooling2D((2, 2)),
    
    Conv2D(64, (3, 3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D((2, 2)),
    
    Conv2D(128, (3, 3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D((2, 2)),

    Conv2D(256, (3, 3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D((2, 2)),
    
    Flatten(),
    Dense(512, activation='relu'),
    BatchNormalization(),
    Dropout(0.5),  # Prevent overfitting

    Dense(21, activation='softmax')  # Assuming 21 classes
])
    
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

In [ ]:
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=9
)

In [ ]:
def preprocess_test_data(test_df, directory):
    """
    Load and preprocess the test data for prediction.
    """
    test_datagen = ImageDataGenerator(rescale=1./255)
    test_generator = test_datagen.flow_from_dataframe(
        dataframe=test_df,
        directory=directory,
        x_col='file_path',
        y_col=None,
        batch_size=32,
        target_size=(128, 128),
        class_mode=None,
        shuffle=False,
        validate_filenames=False # Disable strict filename validation
    )
    return test_generator

In [ ]:
# Load test data
test_dir = '/kaggle/input/bttai-ajl-2025/test/test/'
test_generator = preprocess_test_data(test_df, test_dir)

In [ ]:
predictions = model.predict(test_generator)

In [ ]:
predicted_classes = predictions.argmax(axis=1)
submission_df = test_df[['md5hash']].copy()
submission_df['md5hash'] =  [string[:-4] for string in submission_df['md5hash']]
submission_df['label'] = label_encoder.inverse_transform(predicted_classes)
submission_df[['md5hash', 'label']].to_csv("test_predictions.csv", index=False)